In [0]:
dbutils.widgets.text("secret_scope", "", "Secret scope: ")

In [0]:
scope = dbutils.widgets.get("secret_scope")
storage_account = "dlspl21databricks"
container = "jmizera"

client_id = dbutils.secrets.get(scope, "sp-databricks-adls-appid")
tenant_id = dbutils.secrets.get(scope, "tenant-id")
client_secret = dbutils.secrets.get(scope, "sp-databricks-adls-appkey")

In [0]:
configs = {
  "fs.azure.account.auth.type": "OAuth",
  "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
  "fs.azure.account.oauth2.client.id": client_id,
  "fs.azure.account.oauth2.client.secret": client_secret,
  "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

In [0]:
try:
    dbutils.fs.mount(
    source = f"abfss://{container}@{storage_account}.dfs.core.windows.net/",
    mount_point = f"/mnt/{container}",
    extra_configs = configs
    )
except Exception as e:
    print(e)

In [0]:
for key, value in configs.items():
    spark.conf.set(key, value)

direct_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/"


In [0]:
display(dbutils.fs.ls(direct_path))

I was able to workaround dbutils.fs.mount and reach the storage by passing the credentials to the spark session and accessing it through the abfss path.